# Enhanced Student Performance Prediction Model
### Multi-Model Ensemble with Advanced Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset
df = pd.read_csv('Student_Performance.csv')
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Encode categorical variable
df['Extracurricular Activities'] = (df['Extracurricular Activities'] == 'Yes').astype(int)

# Feature Engineering - Add new calculated features
df['Study_Sleep_Ratio'] = df['Hours Studied'] / (df['Sleep Hours'] + 1)
df['Total_Effort'] = df['Hours Studied'] + df['Sample Question Papers Practiced']
df['Previous_Score_Normalized'] = df['Previous Scores'] / 100
df['Sleep_Efficiency'] = df['Sleep Hours'] * df['Previous Scores'] / 100

print("\nNew features added:")
print(df[['Study_Sleep_Ratio', 'Total_Effort', 'Previous_Score_Normalized', 'Sleep_Efficiency']].head())

In [ ]:
# Prepare features and target
X = df.drop('Performance Index', axis=1)
y = df['Performance Index']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train Multiple Models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
}

results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training {name}...")
    print(f"{'='*60}")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std()
    }
    
    print(f"MAE: {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R² Score: {r2:.4f}")
    print(f"Cross-Val R² (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# Compare Models
results_df = pd.DataFrame(results).T
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(results_df.round(4))

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, metric in enumerate(['R2', 'MAE', 'RMSE']):
    results_df[metric].plot(kind='bar', ax=axes[idx], color=['#667eea', '#764ba2', '#38ef7d'])
    axes[idx].set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].grid(alpha=0.3)
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Select best model (highest R2 score)
best_model_name = results_df['R2'].idxmax()
best_model = models[best_model_name]

print(f"\n🏆 Best Model: {best_model_name}")
print(f"R² Score: {results_df.loc[best_model_name, 'R2']:.4f}")

In [ ]:
# Feature Importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance:")
    print(feature_importance)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='#667eea')
    plt.xlabel('Importance Score')
    plt.title('Feature Importance Analysis', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Save all models
model_package = {
    'best_model': best_model,
    'best_model_name': best_model_name,
    'all_models': models,
    'feature_names': list(X.columns),
    'results': results,
    'scaler': None  # Add if using StandardScaler
}

joblib.dump(model_package, 'enhanced_model.pkl')
print("\n✅ Enhanced model package saved as 'enhanced_model.pkl'")
print(f"Package contains: {best_model_name} + all trained models + metadata")

In [ ]:
# Test prediction with new features
sample_input = pd.DataFrame({
    'Hours Studied': [8],
    'Previous Scores': [85],
    'Extracurricular Activities': [1],
    'Sleep Hours': [7],
    'Sample Question Papers Practiced': [5],
    'Study_Sleep_Ratio': [8/8],
    'Total_Effort': [8+5],
    'Previous_Score_Normalized': [0.85],
    'Sleep_Efficiency': [7*0.85]
})

prediction = best_model.predict(sample_input)[0]
print(f"\nSample Prediction: {prediction:.2f}")
print(f"Model used: {best_model_name}")